In [1]:
from glob import glob
import pandas as pd
from mrmr import mrmr_classif
from scipy.spatial.distance import jensenshannon

#data imports
test = pd.read_csv('data/predictions_test.csv')
train = pd.read_csv('data/predictions_train.csv')
data = test.filter(regex="^src_").select_dtypes("number")
failed = (~test["correct"]).astype(int)

In [2]:
order, relevance, correlations = mrmr_classif(data, failed, K=data.shape[1], return_scores=True, n_jobs=1, show_progress=False)

table = pd.DataFrame({
    "Variable": order,
    "Relevance F": relevance.loc[order].to_numpy(),
    "Redundancy": [correlations.loc[name, order[:i]].mean() for i, name in enumerate(order)]
}, index=pd.RangeIndex(1, len(order) + 1, name="mRMR rank"))

# to rank we first use the best F alone, then we continue the ranking by F divided by redundancy
denominator = table["Redundancy"].replace(1.0, float("inf")).fillna(1.0)
table["Score at selection"] = table["Relevance F"] / denominator

In [3]:
def js_divergence(name):
    p = test[name].value_counts(normalize=True)
    q = train[name].value_counts(normalize=True)
    p, q = p.align(q, join="outer", fill_value=0)
    return jensenshannon(p, q, base=2) ** 2  # Square the distance to get divergence.

table["JS test–train"] = [js_divergence(name) for name in order]
table.round({"Relevance F": 2, "Redundancy": 3, "Score at selection": 2, "JS test–train": 4})

,Variable,Relevance F,Redundancy,Score at selection,JS test–train
mRMR rank,,,,,
1,src_n_vars,186.94,NaN,186.94,0.0001
2,src_n_mul,52.84,0.002,21660.40,0.0000
3,src_n_add,61.96,0.056,1099.86,0.0001
4,src_max_const,42.08,0.245,171.79,0.0005
5,src_n_xor,17.21,0.100,172.79,0.0001
6,src_n_terms,37.69,0.205,184.25,0.0002
7,src_n_or,11.44,0.102,111.72,0.0002
8,src_n_not,6.23,0.152,40.95,0.0002
9,src_max_depth,2.16,0.086,25.07,0.0000


Since in the step1 3d plotting of some selected naive variable we observed significant correlation, we decided that a systemetic approach was necessary, to select informative but at the same time muttually "orthogona" varibles, so that we can reduce the number of variables to visualize in the step3 of projection withiout losign information about the cource of the failure evemts.
We used the so called mRMR method, which is an already available scipy method based on an intuitive idea: a ranking whcih mixes the relevance of each variable (i.e. its informativity about the successful and failed samples) with a redundancy measure of each variable correlation with previously selected variables...the method proceeds iteratively sort of as a GRam-scmidt like orthoganalization prcedure, with the priority initally set by the relevance. So the first variable is chosen only by relevance, while subsequent variables are selected by maximizing the ratio between relevance and redundancy. 
In order to integrate also information about the actual sorce of the failures, we calculate the Jensen-Shannon divergence between the training and test distributions of each variable.